# Inspect VRR preprocessing output

Visualises what `prepare_vrr.py` wrote: per-episode tared force, computed stiffness, virtual-target offsets, and the resulting 19-D action.

Run `prepare_vrr.py --src <raw episodes> --task <task yaml>` first.

Top of cell 1 has the only knobs you need to change.

In [ ]:
# ---- knobs ----
SRC = '../dataset/peg_hole_tac'        # raw episode dir root
VRR_FILE = 'vrr_action.npy'        # per-episode VRR action written by prepare_vrr.py
TARE_FRAMES = 5                    # match prepare_vrr.py default
VRR_PARAMS = dict(k_max=10000.0, k_min=200.0, f_low=0.5, f_high=5.0)
SAMPLE_EPISODES = 4                # how many episodes to plot in detail
# ----------------

import sys, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# the notebook lives in trainflow/notebooks/; repo root is two up.
REPO = Path(os.getcwd()).resolve().parents[1]
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

SRC = (Path(os.getcwd()) / SRC).resolve()
print(f'src       = {SRC}')
print(f'repo root = {REPO}')
assert SRC.is_dir(), f'not a dir: {SRC}'

In [ ]:
# Discover episodes that have both raw inputs and a vrr_action.npy.
eps = []
for d in sorted(SRC.iterdir()):
    if not d.is_dir():
        continue
    if (d / 'eef_state.npy').exists() and (d / 'eef_force.npy').exists() and (d / VRR_FILE).exists():
        eps.append(d)
print(f'{len(eps)} episodes with all 3 files present.')
if not eps:
    raise SystemExit('No episodes — did you run prepare_vrr.py?')

In [ ]:
# Load everything into in-memory dicts keyed by episode index.
# Episodes are small (~100 frames each) so loading the lot is fine.
data = {}
for i, ep in enumerate(eps):
    state = np.load(ep / 'eef_state.npy')                # (T, 7)
    force = np.load(ep / 'eef_force.npy')                # (T, 6)
    action = np.load(ep / VRR_FILE)                      # (T, 19)
    n = min(TARE_FRAMES, len(force))
    baseline = force[:n].mean(axis=0)
    data[i] = dict(
        name=ep.name,
        state=state, force_raw=force, force_tared=force - baseline,
        baseline=baseline, action=action,
    )

T_per_ep = np.array([d['state'].shape[0] for d in data.values()])
print(f'episodes: {len(data)}')
print(f'total frames: {T_per_ep.sum()}')
print(f'T per ep    : min={T_per_ep.min()}  median={int(np.median(T_per_ep))}  max={T_per_ep.max()}')

## 1. Per-episode force baselines (validates the drift hypothesis)

If the F/T sensor drifts between sessions, the mean of the first N frames of each episode should move with episode index. A flat horizontal line = no drift; a trend or step = drift.

In [ ]:
baselines = np.stack([d['baseline'] for d in data.values()])    # (n_ep, 6)
fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharex=True)
labels = ['Fx', 'Fy', 'Fz', 'Tx', 'Ty', 'Tz']
for ax, col, lab in zip(axes.flat, range(6), labels):
    ax.plot(baselines[:, col], marker='o', ms=3, lw=0.8)
    ax.axhline(0, color='k', lw=0.5, alpha=0.5)
    ax.axhline(baselines[:, col].mean(), color='C1', lw=0.8, ls='--',
               label=f'mean={baselines[:, col].mean():.2f}')
    ax.set_title(f'{lab}  (std={baselines[:, col].std():.2f})')
    ax.set_xlabel('episode index')
    ax.set_ylabel(f'{lab} (N or N·m)')
    ax.legend(loc='best', fontsize=8)
fig.suptitle(f'Per-episode force baseline (mean of first {TARE_FRAMES} frames)', y=1.02)
fig.tight_layout()
plt.show()

## 2. Concatenated raw vs tared force across all episodes

All episodes laid end-to-end as one continuous timeline. Three stacked subplots, one per linear-force axis (Fx, Fy, Fz). Each subplot overlays raw and tared traces. The raw trace should sit on a constant non-zero band on the axis carrying the gravity/mounting bias (typically Z) and near zero on the others; the tared trace should hover near zero on all axes in free space and spike during contact. Faint vertical lines mark episode boundaries.

In [ ]:
# Concatenate every episode's raw and tared linear-force across all 3 axes.
raw   = np.concatenate([d['force_raw'][:, :3]   for d in data.values()], axis=0)
tared = np.concatenate([d['force_tared'][:, :3] for d in data.values()], axis=0)
ep_ends = np.cumsum([d['force_raw'].shape[0] for d in data.values()])

axis_labels = ['Fx', 'Fy', 'Fz']
fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
t = np.arange(raw.shape[0])
for i, (ax, lab) in enumerate(zip(axes, axis_labels)):
    ax.plot(t, raw[:, i],   color='C3', lw=0.6, label=f'raw {lab}')
    ax.plot(t, tared[:, i], color='C0', lw=0.6, label=f'tared {lab}')
    for x in ep_ends[:-1]:
        ax.axvline(x, color='k', lw=0.3, alpha=0.2)
    ax.axhline(0, color='k', lw=0.4, alpha=0.4)
    ax.set_ylabel(f'{lab} (N)')
    ax.legend(loc='upper right', fontsize=8, ncol=2)
axes[0].set_title(f'Raw vs tared force per axis — {len(data)} episodes, {raw.shape[0]} frames')
axes[-1].set_xlabel('frame (all episodes concatenated)')
axes[0].set_xlim(0, raw.shape[0])
fig.tight_layout(); plt.show()

print('per-axis summary (raw / tared):')
for i, lab in enumerate(axis_labels):
    print(f'  {lab}: raw   mean={raw[:, i].mean():+7.2f}  std={raw[:, i].std():.2f}  '
          f'range=[{raw[:, i].min():+7.2f}, {raw[:, i].max():+7.2f}]')
    print(f'      tared mean={tared[:, i].mean():+7.2f}  std={tared[:, i].std():.2f}  '
          f'range=[{tared[:, i].min():+7.2f}, {tared[:, i].max():+7.2f}]')


## 3. Raw vs tared force — one sample episode

Confirms the tare moves free-space frames toward zero. If you see post-tare frames still hovering far from zero in the *first few* timesteps, your start-of-episode wasn't in free space and the tare baseline is biased.

In [ ]:
rng = np.random.default_rng(0)
picked = rng.choice(len(data), size=min(SAMPLE_EPISODES, len(data)), replace=False)
picked = sorted(picked.tolist())
print(f'Showing episodes {picked}')

In [ ]:
for idx in picked:
    d = data[idx]
    fig, axes = plt.subplots(2, 1, figsize=(11, 4.5), sharex=True)
    t = np.arange(d['state'].shape[0])
    for col, lab in zip(range(3), ['Fx', 'Fy', 'Fz']):
        axes[0].plot(t, d['force_raw'][:, col], label=lab)
        axes[1].plot(t, d['force_tared'][:, col], label=lab)
    axes[0].set_title(f"ep {idx} ({d['name']}): raw force")
    axes[1].set_title('tared force')
    axes[0].axhline(0, color='k', lw=0.4, alpha=0.4)
    axes[1].axhline(0, color='k', lw=0.4, alpha=0.4)
    for ax in axes: ax.legend(loc='best', fontsize=8); ax.set_ylabel('force (N)')
    axes[1].set_xlabel('frame')
    fig.tight_layout()
    plt.show()

## 4. Force magnitude + stiffness over time

Stiffness should drop where force magnitude rises, riding the piecewise-linear ramp between `f_low` and `f_high`.

In [ ]:
for idx in picked:
    d = data[idx]
    mag = np.linalg.norm(d['force_tared'][:, :3], axis=-1)
    stiffness = d['action'][:, 18]   # last dim of the 19-D action
    t = np.arange(len(mag))
    fig, ax1 = plt.subplots(figsize=(11, 3.3))
    ax1.plot(t, mag, color='C0', label='|F|')
    ax1.axhline(VRR_PARAMS['f_low'], color='C0', ls=':', lw=0.7, alpha=0.7, label=f"f_low={VRR_PARAMS['f_low']}")
    ax1.axhline(VRR_PARAMS['f_high'], color='C0', ls='--', lw=0.7, alpha=0.7, label=f"f_high={VRR_PARAMS['f_high']}")
    ax1.set_ylabel('|F| (N)')
    ax2 = ax1.twinx()
    ax2.plot(t, stiffness, color='C3', label='stiffness')
    ax2.set_ylabel('stiffness (N/m)')
    ax2.set_ylim(0, VRR_PARAMS['k_max'] * 1.05)
    ax1.set_title(f"ep {idx} ({d['name']}): force magnitude vs stiffness")
    ax1.set_xlabel('frame')
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, loc='best', fontsize=8)
    fig.tight_layout(); plt.show()

## 5. Force-magnitude → stiffness mapping (sanity check)

Should trace the piecewise-linear curve exactly: flat at `k_max` below `f_low`, ramp down to `k_min`, flat at `k_min` above `f_high`.

In [ ]:
mag_all = []
k_all = []
for d in data.values():
    mag_all.append(np.linalg.norm(d['force_tared'][:, :3], axis=-1))
    k_all.append(d['action'][:, 18])
mag_all = np.concatenate(mag_all)
k_all = np.concatenate(k_all)

# reference curve
f_low, f_high = VRR_PARAMS['f_low'], VRR_PARAMS['f_high']
k_max, k_min = VRR_PARAMS['k_max'], VRR_PARAMS['k_min']
x = np.linspace(0, max(f_high * 1.3, mag_all.max()), 400)
y = np.where(x < f_low, k_max,
             np.where(x > f_high, k_min,
                      k_max - (k_max - k_min) * (x - f_low) / (f_high - f_low)))

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(mag_all, k_all, s=2, alpha=0.25, label='data')
ax.plot(x, y, 'r-', lw=1.2, label='expected curve')
ax.axvline(f_low, color='k', ls=':', lw=0.7, alpha=0.5)
ax.axvline(f_high, color='k', ls='--', lw=0.7, alpha=0.5)
ax.set_xlabel('|F| post-tare (N)')
ax.set_ylabel('stiffness (N/m)')
ax.set_title(f'{len(mag_all)} frames across {len(data)} episodes')
ax.legend(); fig.tight_layout(); plt.show()

## 6. Virtual-target offset magnitude over time

Offset from real TCP to the implied impedance attractor:  `||x_v - x_real|| = ||F/K||`. Should spike during contact, hover near zero in free space.

In [ ]:
for idx in picked:
    d = data[idx]
    real_xyz = d['action'][:, :3]
    virt_xyz = d['action'][:, 9:12]
    off = np.linalg.norm(virt_xyz - real_xyz, axis=-1) * 1000  # mm
    mag = np.linalg.norm(d['force_tared'][:, :3], axis=-1)
    t = np.arange(len(off))
    fig, ax = plt.subplots(figsize=(11, 3.0))
    ax.plot(t, off, color='C2', label='||x_v - x_real|| (mm)')
    ax2 = ax.twinx()
    ax2.plot(t, mag, color='C0', alpha=0.5, lw=0.8, label='|F| (N)')
    ax.set_ylabel('virtual offset (mm)'); ax2.set_ylabel('|F| (N)')
    ax.set_xlabel('frame')
    ax.set_title(f"ep {idx} ({d['name']}): virtual offset vs force")
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc='best', fontsize=8)
    fig.tight_layout(); plt.show()

## 7. Action dim summary

Distribution of every dimension of the 19-D action across all frames. Useful for spotting degenerate dims (e.g. constant stiffness).

In [ ]:
all_actions = np.concatenate([d['action'] for d in data.values()], axis=0)
print(f'all_actions shape: {all_actions.shape}')

labels = (['x', 'y', 'z'] + [f'rot6d_{i}' for i in range(6)] +
          ['v_x', 'v_y', 'v_z'] + [f'v_rot6d_{i}' for i in range(6)] +
          ['stiffness'])
fig, axes = plt.subplots(4, 5, figsize=(15, 9))
for i, ax in enumerate(axes.flat):
    if i >= 19:
        ax.axis('off'); continue
    a = all_actions[:, i]
    ax.hist(a, bins=40)
    ax.set_title(f'{i}: {labels[i]}  μ={a.mean():.3g}  σ={a.std():.3g}', fontsize=8)
    ax.tick_params(labelsize=7)
fig.tight_layout(); plt.show()

## 8. Real vs virtual TCP trajectory — one sample episode (3D)

Sanity check that the virtual target tracks the real TCP with offsets only during contact.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

idx = picked[0]
d = data[idx]
real_xyz = d['action'][:, :3]
virt_xyz = d['action'][:, 9:12]

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot(real_xyz[:, 0], real_xyz[:, 1], real_xyz[:, 2], 'C0-', lw=1.0, label='real TCP')
ax.plot(virt_xyz[:, 0], virt_xyz[:, 1], virt_xyz[:, 2], 'C3-', lw=1.0, label='virtual target')
ax.scatter(*real_xyz[0], color='C0', s=30, label='real start')
ax.scatter(*real_xyz[-1], color='C0', marker='x', s=30, label='real end')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title(f"ep {idx} ({d['name']}): real vs virtual TCP")
ax.legend(fontsize=8)
fig.tight_layout(); plt.show()